In [0]:

import pandas as pd


# Carregar os dados com Pandas
df = pd.read_csv("/Volumes/workspace/default/dados_entregas/raw/E-commerce_Delivery_Shipping_Data_2026.csv")

#VERIFICAR DATASET
display(df.info())
display(df.head())


#VERIFICAR DUPLICADAS
duplicatas_linhas = df.duplicated().sum()
print("Linhas totalmente duplicadas:", duplicatas_linhas)

#VERIFICAR DUPLICADAS POR ID
duplicatas_pedidos = df["order_id"].duplicated().sum()
print("IDs de pedidos duplicados:", duplicatas_pedidos)

#VERIFICAR TYPES
df.dtypes


In [0]:


#TRATAR TYPE DA COLUNA ORDER_DATE PARA DATA
df['order_date'] = pd.to_datetime(df['order_date']).dt.date



#VERIFICAR RELACAO ENTRE COLUNAS RETURN_REQUESTED E RETURN_REASON PARA TRATAR NULOS
display(
    df.groupby("return_requested")["return_reason"].count()
)

#NULOS IDENTIFICADOS POR AUSENCIA DE SOLICITACAO DE DEVOLUCAO
#TRATAR NULOS
df["return_reason"] = df["return_reason"].fillna("N/A")
df.info()

# VERIFICAR CONSISTENCIA DAS COLUNAS DE PRAZOS
print("Valores negativos:",((df['delivery_delay_days'] < 0).any()))


display(
    df[[
        "promised_delivery_days",
        "actual_delivery_days",
        "delivery_delay_days"
    ]].head(20)
)

# COMPARAR ATRASO REGISTRADO COM ATRASO CALCULADO
atraso_calculado = (
            df["actual_delivery_days"] - df["promised_delivery_days"]
        ).clip(lower=0)
(df["delivery_delay_days"] != atraso_calculado).sum()

# VERIFICAR DE O INDICADOR DE ATRASO DA COLUNA late_delivery ESTA CORRETO
display(
    df[[
        "delivery_delay_days",
        "late_delivery"
    ]].head(15)
)
# VERIFICAR TODA A COLUNA  
df.loc[
    ((df["delivery_delay_days"] > 0) & (df["late_delivery"] != "Yes")) |
    ((df["delivery_delay_days"] <= 0) & (df["late_delivery"] != "No")),
    ["delivery_delay_days", "late_delivery"]
]


In [0]:
# EXPORTAR DF PARA CSV
df.to_csv(
    "/Volumes/workspace/default/dados_entregas/silver/entregas_tratadas.csv",
    index=False
)